# 02 — Google Client

All Google API interactions live in `src/google_client.py`.

## Service Account setup

1. [Google Cloud Console](https://console.cloud.google.com) → create a project  
2. Enable **Drive API**, **Sheets API**, **Docs API**  
3. IAM & Admin → Service Accounts → Create → download JSON key → save as `service_account.json` in project root  
4. Share your Lab Assistant Drive folder with the service account email  
5. Share your Lab Assistant Sheets file with the same email

Set in `.env`:
```
GOOGLE_SERVICE_ACCOUNT_FILE=service_account.json
DRIVE_PROTOCOLS_FOLDER_ID=<folder id from Drive URL>
DRIVE_SESSION_REPORTS_FOLDER_ID=<folder id>
SHEETS_SPREADSHEET_ID=<spreadsheet id>
```

In [ ]:
import sys, asyncio
sys.path.insert(0, '..')

from src.google_client import (
    list_protocols, download_docx, find_companion_doc_id,
    get_doc_text, append_doc_text, create_session_doc, get_doc_url,
    read_sheet, append_sheet_row,
)
from src.config import SHEET_LAB_JOURNAL
print('Imports OK')

## 1. List protocols from Drive

In [ ]:
protocols = asyncio.run(list_protocols())
print(f'Found {len(protocols)} protocol(s):')
for p in protocols:
    print(f"  {p['name']}  (modified {p['modifiedTime'][:10]})")

## 2. Download a protocol and check size

In [ ]:
import os
if protocols:
    p = protocols[0]
    data = asyncio.run(download_docx(p['id']))
    print(f"Downloaded '{p['name']}': {len(data):,} bytes")

## 3. Companion knowledge doc lookup

Expected naming: if protocol is `Western_Blot_v3.docx`, companion should be a Google Doc named `Western_Blot_v3_context`.

In [ ]:
if protocols:
    stem = os.path.splitext(protocols[0]['name'])[0]
    doc_id = asyncio.run(find_companion_doc_id(stem))
    if doc_id:
        text = asyncio.run(get_doc_text(doc_id))
        print(f'Companion: {doc_id}  ({len(text)} chars)\n{text[:400]}')
    else:
        print(f'No companion doc found for "{stem}_context".')

## 4. Read Lab Journal sheet

In [ ]:
rows = asyncio.run(read_sheet(SHEET_LAB_JOURNAL))
print(f'{len(rows)} row(s) in Lab Journal')
for r in rows[:3]:
    print(' | '.join(str(c) for c in r))

## 5. Test session doc creation

⚠️ Creates a real Google Doc in Session Reports folder. Delete it afterwards.

In [ ]:
# Uncomment to test:
# doc_id = asyncio.run(create_session_doc('TEST — Delete me'))
# asyncio.run(append_doc_text(doc_id, 'Test entry.'))
# print(get_doc_url(doc_id))